# Speech Recognition Directory Retrieval

## Project Overview
This system retrieves telephone directory information by recognizing **spelled names** (e.g., "S-M-I-T-H"). 

### Technologies Used
* **SpeechRecognition:** Google Speech Recognition API for converting audio to text.
* **FuzzyWuzzy:** For approximate string matching (correcting "F" vs "S" errors).
* **JavaScript Integration:** To enable microphone recording within the Google Colab environment.

## Step 1: Install Dependencies
We need to install system-level dependencies for audio processing (`portaudio`) before installing the Python libraries.

In [ ]:
# 1. Install system dependencies for audio
!apt-get update
!apt-get install -y portaudio19-dev ffmpeg

# 2. Install Python libraries
!pip install SpeechRecognition pyaudio fuzzywuzzy python-Levenshtein

## Step 2: Setup Directory & Helper Functions
Here we define our phonebook database and the logic to clean up the spelled text.

In [ ]:
import speech_recognition as sr
from fuzzywuzzy import process
import os

# --- CONFIGURATION ---
directory = {
    "SMITH": "555-0101",
    "JOHNSON": "555-0123",
    "WILLIAMS": "555-0145",
    "JONES": "555-0167",
    "BROWN": "555-0189",
    "DAVIS": "555-0201",
    "MILLER": "555-0234",
    "WILSON": "555-0256",
    "MOORE": "555-0278",
    "TAYLOR": "555-0300"
}

def clean_spelled_input(text):
    """
    Converts recognized text into a single string.
    Example: "S M I T H" -> "SMITH"
    """
    if not text:
        return ""
    return text.replace(" ", "").replace("-", "").upper()

def retrieve_info(spelled_name):
    """
    Uses Fuzzy Matching to find the closest name in the directory.
    """
    query = clean_spelled_input(spelled_name)
    print(f"Processing query: {query}")
    
    # Get matches
    all_names = list(directory.keys())
    best_match, score = process.extractOne(query, all_names)
    
    print(f"Best Match: {best_match} (Confidence: {score}%)")
    
    # Confidence Threshold (80%)
    if score >= 80:
        return directory[best_match], best_match
    return None, None

## Step 3: Browser-Based Audio Recorder
Since standard microphone access doesn't work in cloud notebooks, we use this JavaScript snippet to record directly from your browser.

In [ ]:
from IPython.display import HTML, Audio, display
from google.colab.output import eval_js
from base64 import b64decode

AUDIO_HTML = """
<script>
var my_media_recorder = null;
var my_audio_chunks = [];
var my_resolve = null;
var my_reject = null;

function _startRecording() {
    navigator.mediaDevices.getUserMedia({ audio: true }).then(function(stream) {
        my_media_recorder = new MediaRecorder(stream);
        my_audio_chunks = [];
        my_media_recorder.ondataavailable = function(event) {
            my_audio_chunks.push(event.data);
        };
        my_media_recorder.onstop = function() {
            var blob = new Blob(my_audio_chunks, { 'type' : 'audio/ogg; codecs=opus' });
            var reader = new FileReader();
            reader.onload = function(event) {
                my_resolve(event.target.result.split(',')[1]); 
            };
            reader.readAsDataURL(blob);
        };
        my_media_recorder.start();
        document.getElementById('status').innerText = 'Recording... Speak now (e.g. S-M-I-T-H)!';
    }).catch(function(err) {
        my_reject(err);
    });
}

function _stopRecording() {
    if (my_media_recorder && my_media_recorder.state === 'recording') {
        my_media_recorder.stop();
        document.getElementById('status').innerText = 'Processing...';
    }
}

function record_audio() {
    return new Promise(function(resolve, reject) {
        my_resolve = resolve;
        my_reject = reject;
        _startRecording();
    });
}
</script>
<div style='border:1px solid #ccc; padding:10px; border-radius:5px;'>
    <button onclick="record_audio()" style='background-color:#4CAF50; color:white; padding:10px; border:none; border-radius:3px;'>Start Recording</button>
    <button onclick="_stopRecording()" style='background-color:#f44336; color:white; padding:10px; border:none; border-radius:3px; margin-left:10px;'>Stop Recording</button>
    <p id="status" style='margin-top:10px; font-weight:bold;'>Ready</p>
</div>
"""

def record_voice(filename="input_name.wav"):
    display(HTML(AUDIO_HTML))
    data = eval_js('record_audio()')
    binary = b64decode(data)
    
    # Save raw audio to file (usually webm/ogg from browser)
    with open("temp_audio.webm", 'wb') as f:
        f.write(binary)
    
    # Convert to proper WAV format for SpeechRecognition using ffmpeg
    !ffmpeg -y -i temp_audio.webm -ac 1 -ar 16000 {filename} > /dev/null 2>&1
    print(f"Audio recorded and saved to {filename}")

## Step 4: Main Execution
Run this block to:  
1. **Record** your voice.  
2. **Transcribe** the audio using Google Speech API.  
3. **Match** the name and retrieve the number.

In [ ]:
def main():
    audio_file = "input_name.wav"
    
    # 1. Capture Audio
    print("--- STEP 1: RECORDING ---")
    record_voice(audio_file)
    
    # 2. Process Audio
    print("\n--- STEP 2: PROCESSING ---")
    recognizer = sr.Recognizer()
    
    try:
        with sr.AudioFile(audio_file) as source:
            audio_data = recognizer.record(source)
            
        # Transcribe
        text = recognizer.recognize_google(audio_data)
        print(f"Recognized Text: '{text}'")
        
        # 3. Retrieve Info
        print("\n--- STEP 3: RETRIEVAL ---")
        number, name = retrieve_info(text)
        
        if number:
            print(f"\n>>> SUCCESS! Found: {name}")
            print(f">>> Phone Number: {number}")
        else:
            print("\n>>> FAILURE: Name not found in directory.")
            
    except sr.UnknownValueError:
        print("ERROR: Google Speech Recognition could not understand audio.")
    except sr.RequestError as e:
        print(f"ERROR: Could not request results from Google Speech API; {e}")
    except Exception as e:
        print(f"ERROR: {e}")

if __name__ == "__main__":
    main()